In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime, gc
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, ArrayType, DoubleType

# Cấu hình đường dẫn chuẩn của Leader
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_FILE = BASE_PATH + "processed/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs/candidates/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Khởi tạo Spark tối ưu 10GB RAM
spark = SparkSession.builder \
    .appName("HM_Weekly_Trending_Pipeline") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "10g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print("✅ Spark Ready! Hệ thống đã sẵn sàng tạo 'Phao cứu sinh'.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Spark Ready! Hệ thống đã sẵn sàng tạo 'Phao cứu sinh'.


In [ ]:
# 1. Đọc dữ liệu giao dịch
transactions = spark.read.parquet(INPUT_FILE)

# 2. Tính toán mốc thời gian (Logic 6-1-1)
max_date = transactions.select(F.max("t_dat")).collect()[0][0]
test_start_date = max_date - datetime.timedelta(days=7)         # Đầu Tuần 8
val_start_date = test_start_date - datetime.timedelta(days=7)   # Đầu Tuần 7 (Validation)
w6_start = val_start_date - datetime.timedelta(days=7)          # Đầu Tuần 6 (Dùng để tính Trending)

print(f"📅 Ngày cuối cùng dữ liệu: {max_date}")
print(f"📊 Tính Trending dựa trên Tuần 6: {w6_start} -> {val_start_date}")
print(f"🎯 Mục tiêu dự báo cho Tuần 7: {val_start_date} -> {test_start_date}")

📅 Ngày cuối cùng dữ liệu: 2020-09-21 17:00:00
📊 Tính Trending dựa trên Tuần 6: 2020-08-31 17:00:00 -> 2020-09-07 17:00:00
🎯 Mục tiêu dự báo cho Tuần 7: 2020-09-07 17:00:00 -> 2020-09-14 17:00:00


In [ ]:
# 1. Lọc giao dịch trong Tuần 6
weekly_sales = transactions.filter(
    (F.col("t_dat") >= F.lit(w6_start)) &
    (F.col("t_dat") < F.lit(val_start_date))
)

# 2. Đếm và lấy Top 12 (Đảm bảo ID 10 chữ số)
top_12_trending = weekly_sales.groupBy("article_id") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(12) \
    .withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0"))

# Chuyển thành Python List để chuẩn bị phân phối cho tất cả User
trending_list = [row['article_id'] for row in top_12_trending.collect()]

print(f"🔥 Danh sách 12 món Trending 'vàng':")
for i, item in enumerate(trending_list):
    print(f"   Rank {i+1}: {item}")

🔥 Danh sách 12 món Trending 'vàng':
   Rank 1: 0915526001
   Rank 2: 0751471001
   Rank 3: 0751471043
   Rank 4: 0919365008
   Rank 5: 0706016001
   Rank 6: 0915529003
   Rank 7: 0918292001
   Rank 8: 0863595006
   Rank 9: 0896152002
   Rank 10: 0898694001
   Rank 11: 0797988002
   Rank 12: 0448509014


In [ ]:
# 1. Lấy danh sách khách hàng cần dự báo (Tất cả khách xuất hiện trong dữ liệu)
# Lưu ý: Bạn có thể lấy từ file sample_submission nếu muốn bao phủ cả khách mới hoàn toàn
customers_df = transactions.select("customer_id").distinct()

# 2. Gán mảng 12 món Trending cho mỗi khách hàng
# Sử dụng lit() để biến các giá trị string thành cột Spark
trending_candidates_df = customers_df.withColumn(
    "trending_candidates",
    F.array([F.lit(x) for x in trending_list])
)

# 3. Lưu file Parquet để phục vụ tầng Ranking (XGBoost)
trending_candidates_df.write.mode("overwrite").parquet(OUTPUT_DIR + "trending_candidates_W7.parquet")

print(f"✅ Đã tạo xong file ứng viên Trending cho {trending_candidates_df.count():,} khách hàng.")
print(f"📍 File lưu tại: {OUTPUT_DIR}trending_candidates_W7.parquet")

✅ Đã tạo xong file ứng viên Trending cho 1,362,281 khách hàng.
📍 File lưu tại: /content/drive/MyDrive/HM-DATA/outputs/candidates/trending_candidates_W7.parquet


In [ ]:
# 1. Ground Truth: Thực tế khách mua ở Tuần 7
ground_truth_w7 = transactions.filter(
    (F.col("t_dat") >= F.lit(val_start_date)) &
    (F.col("t_dat") < F.lit(test_start_date))
).select("customer_id", F.lpad(F.col("article_id").cast("string"), 10, "0").alias("article_id"))

actual_counts = ground_truth_w7.groupBy("customer_id").count().withColumnRenamed("count", "actual_cnt")

# 2. Explode danh sách 12 món để tính Hits
candidates_exploded = trending_candidates_df.select(
    "customer_id",
    F.explode("trending_candidates").alias("article_id")
)

# 3. Join tìm món trùng khớp
hits = ground_truth_w7.join(candidates_exploded, ["customer_id", "article_id"], "inner") \
    .groupBy("customer_id").count().withColumnRenamed("count", "hit_cnt")

# 4. Tính Recall trung bình
recall_stats = actual_counts.join(hits, "customer_id", "left").fillna(0)
final_recall_trending = recall_stats.select(F.avg(F.col("hit_cnt") / F.col("actual_cnt"))).collect()[0][0]

print("-" * 50)
print(f"📊 KẾT QUẢ RECALL NHÁNH TRENDING (N=12)")
print(f"Average Recall@12: {final_recall_trending:.6f}")
print("-" * 50)

--------------------------------------------------
📊 KẾT QUẢ RECALL NHÁNH TRENDING (N=12)
Average Recall@12: 0.023231
--------------------------------------------------
